In [2]:
"""
Fetch Solana yield pools from DefiLlama and save to CSV.

Run manually:
    python solana_pools.py

Or schedule it to refresh automatically (see notes at the bottom of this file).
"""

import csv
import requests
from datetime import datetime, timezone

URL = "https://yields.llama.fi/pools"
OUTPUT_FILE = "solana_pools.csv"

COLUMNS = [
    "Project",
    "Symbol",
    "APY",
    "APY Base",
    "APY Reward",
    "Reward Tokens",
    "Pool ID",
    "APY % 1D",
    "APY % 7D",
    "APY % 30D",
    "Stablecoin",
    "APY Base 7D",
    "TVL USD",
    "Underlying Tokens",
]


def get_solana_pools():
    response = requests.get(URL, timeout=30)
    response.raise_for_status()
    pools = response.json().get("data", [])

    rows = []
    for pool in pools:
        if pool.get("chain") == "Solana":
            rows.append({
                "Project": pool.get("project", ""),
                "Symbol": pool.get("symbol", ""),
                "APY": pool.get("apy", 0) or 0,
                "APY Base": pool.get("apyBase", 0) or 0,
                "APY Reward": pool.get("apyReward", 0) or 0,
                "Reward Tokens": ", ".join(pool.get("rewardTokens") or []),
                "Pool ID": pool.get("pool", ""),
                "APY % 1D": pool.get("apyPct1D", 0) or 0,
                "APY % 7D": pool.get("apyPct7D", 0) or 0,
                "APY % 30D": pool.get("apyPct30D", 0) or 0,
                "Stablecoin": pool.get("stablecoin", False),
                "APY Base 7D": pool.get("apyBase7d", 0) or 0,
                "TVL USD": pool.get("tvlUsd", 0) or 0,
                "Underlying Tokens": ", ".join(pool.get("underlyingTokens") or []),
            })

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(rows)

    print(f"[{datetime.now(timezone.utc).isoformat()}] "
          f"Wrote {len(rows)} Solana pools to {OUTPUT_FILE}")


if __name__ == "__main__":
    get_solana_pools()

# -----------------------------------------------------------------------
# Automating the refresh (pick one):
#
# 1) Windows Task Scheduler (simplest, no cloud needed):
#    - Create a Basic Task -> Trigger: Daily / hourly -> Action: Start a program
#    - Program: path to python.exe, Arguments: full path to this script
#
# 2) GitHub Actions (if you want the CSV hosted/versioned in a repo,
#    matches the automation pattern you already use for other pipelines):
#    - Put this script in a repo, add a workflow with a `schedule: cron`
#      trigger, run the script, then commit the updated CSV back
#      (or upload it as an artifact) each run.
# -----------------------------------------------------------------------

[2026-07-25T16:24:16.629394+00:00] Wrote 2901 Solana pools to solana_pools.csv


In [6]:
"""
Load solana_pools.csv into SQLite and let you query it with SQL.

Image + a general App Link are now filled in AUTOMATICALLY per project,
pulled from DefiLlama's own public protocols list (same "project" slug
your pools already use) - no manual typing needed for those two.

Pool Address is the one thing that still has to be entered by hand
(same as your Mantle Clearpool/Fluxion mappings) - no public API
exposes on-chain pool addresses. You can also override the
auto Image/App Link per-pool in pool_metadata.csv if a specific pool
needs something more precise than the general project homepage.

Flow:
  1. solana_pools.py refreshes solana_pools.csv (raw DefiLlama data).
  2. This script loads that CSV into the `pools` table every run.
  3. `project_lookup` is fetched fresh every run from DefiLlama's
     protocols API - Image + App Link per project, fully automatic.
  4. Your manual Pool Address (+ optional overrides) live in
     `pool_metadata`, seeded from pool_metadata.csv the FIRST time
     only, then left alone.
  5. `pools_enriched` is a view joining pools + project_lookup +
     pool_metadata, with manual values winning if present.

Usage:
    python solana_pools_db.py                  # load/refresh + rebuild view
    sqlite3 solana_pools.db                     # open a shell
    sqlite> SELECT * FROM pools_enriched WHERE "TVL USD" > 0 ORDER BY Project, "TVL USD" DESC;
"""

import csv
import os
import sqlite3
import requests

DB_FILE = "solana_pools.db"
POOLS_CSV = "solana_pools.csv"
METADATA_CSV = "pool_metadata.csv"  # you maintain this by hand (Pool Address, overrides)
PROTOCOLS_URL = "https://api.llama.fi/protocols"


def load_pools(conn):
    """(Re)load raw pool data. Safe to overwrite - it's just a refresh."""
    with open(POOLS_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        columns = reader.fieldnames

    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS pools")
    col_defs = ", ".join(f'"{c}" TEXT' for c in columns)
    cur.execute(f"CREATE TABLE pools ({col_defs})")

    placeholders = ", ".join("?" for _ in columns)
    cur.executemany(
        f"INSERT INTO pools VALUES ({placeholders})",
        [[row[c] for c in columns] for row in rows],
    )
    conn.commit()
    print(f"Loaded {len(rows)} rows into 'pools'.")


def load_project_lookup(conn):
    """
    Auto-populate Image + general App Link per project from DefiLlama's
    public protocols list - no manual typing needed. Refreshed every run.
    """
    resp = requests.get(PROTOCOLS_URL, timeout=30)
    resp.raise_for_status()
    protocols = resp.json()

    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS project_lookup")
    cur.execute(
        """
        CREATE TABLE project_lookup (
            "Project" TEXT PRIMARY KEY,
            "Image" TEXT,
            "App Link" TEXT
        )
        """
    )

    rows = []
    for p in protocols:
        slug = p.get("slug") or p.get("module")
        if not slug:
            continue
        rows.append((slug, p.get("logo", ""), p.get("url", "")))

    cur.executemany("INSERT OR REPLACE INTO project_lookup VALUES (?, ?, ?)", rows)
    conn.commit()
    print(f"Loaded {len(rows)} projects into 'project_lookup' (auto, from DefiLlama).")


def ensure_metadata_table(conn):
    """
    Create pool_metadata once. This now only holds Pool Address (never
    available from any public API - has to be sourced/entered by hand,
    same as your Mantle Clearpool/Fluxion mappings) plus optional
    per-pool overrides for Image/App Link if the auto project-level
    ones aren't specific enough for a given pool.
    """
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS pool_metadata (
            "Pool ID" TEXT PRIMARY KEY,
            "Pool Address" TEXT,
            "Image" TEXT,
            "App Link" TEXT
        )
        """
    )
    conn.commit()

    cur.execute("SELECT COUNT(*) FROM pool_metadata")
    is_empty = cur.fetchone()[0] == 0

    if is_empty and os.path.exists(METADATA_CSV):
        with open(METADATA_CSV, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = [
                (r["Pool ID"], r.get("Pool Address", ""), r.get("Image", ""), r.get("App Link", ""))
                for r in reader
            ]
        cur.executemany(
            'INSERT OR IGNORE INTO pool_metadata VALUES (?, ?, ?, ?)', rows
        )
        conn.commit()
        print(f"Seeded pool_metadata with {len(rows)} rows from {METADATA_CSV}.")
    elif is_empty:
        # Create an empty template CSV so you know the expected columns.
        with open(METADATA_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["Pool ID", "Pool Address", "Image", "App Link"])
        print(f"No metadata yet - created a blank template at {METADATA_CSV}.")


def create_enriched_view(conn):
    cur = conn.cursor()
    cur.execute("DROP VIEW IF EXISTS pools_enriched")
    cur.execute(
        """
        CREATE VIEW pools_enriched AS
        SELECT
            p.*,
            m."Pool Address",
            COALESCE(m."Image", pl."Image") AS "Image",
            COALESCE(m."App Link", pl."App Link") AS "App Link"
        FROM pools p
        LEFT JOIN project_lookup pl ON p."Project" = pl."Project"
        LEFT JOIN pool_metadata m ON p."Pool ID" = m."Pool ID"
        """
    )
    conn.commit()


def main():
    conn = sqlite3.connect(DB_FILE)
    load_pools(conn)
    load_project_lookup(conn)
    ensure_metadata_table(conn)
    create_enriched_view(conn)
    conn.close()
    print(f"Ready. Query {DB_FILE} -> table 'pools_enriched'.")


if __name__ == "__main__":
    main()

# -----------------------------------------------------------------------
# Image + App Link are now automatic (from DefiLlama's protocols list),
# refreshed every run in `project_lookup`.
#
# Only Pool Address needs manual entry, plus optional per-pool overrides
# for Image/App Link. To add these, either:
#
#   A) Edit rows directly:
#      sqlite3 solana_pools.db
#      sqlite> INSERT OR REPLACE INTO pool_metadata VALUES
#          ('d8733ab8-a147-4e31-a668-2c9dff24ea56',   -- Pool ID
#           '9e709e57...',                              -- Pool Address
#           NULL,                                       -- Image override (optional)
#           NULL);                                      -- App Link override (optional)
#
#   B) Edit pool_metadata.csv (Pool ID, Pool Address, Image, App Link columns) and
#      re-seed by clearing the table, then re-running this script:
#      sqlite> DELETE FROM pool_metadata;
#
# `pools_enriched` always reflects: manual override (if set) > auto
# project-level Image/App Link > NULL.
# -----------------------------------------------------------------------

Loaded 2877 rows into 'pools'.
Loaded 7942 projects into 'project_lookup' (auto, from DefiLlama).
Seeded pool_metadata with 0 rows from pool_metadata.csv.
Ready. Query solana_pools.db -> table 'pools_enriched'.


In [2]:
import sqlite3
conn = sqlite3.connect("solana_pools.db")
conn.execute("DROP TABLE IF EXISTS pool_metadata")
conn.commit()
conn.close()

In [8]:
import sqlite3

conn = sqlite3.connect("solana_pools.db")
conn.row_factory = sqlite3.Row  # <-- this line does it
cur = conn.cursor()

cur.execute("SELECT * FROM pools_enriched LIMIT 5")
for row in cur.fetchall():
    print(dict(row))

{'Project': 'binance-staked-sol', 'Symbol': 'BNSOL', 'APY': '4.7782', 'APY Base': '4.7782', 'APY Reward': '0', 'Reward Tokens': '', 'Pool ID': '9e709e57-84eb-496b-82ce-2e8f6a17db1b', 'APY % 1D': '-0.06135', 'APY % 7D': '-0.1271', 'APY % 30D': '-0.31119', 'Stablecoin': 'False', 'APY Base 7D': '0', 'TVL USD': '756970273', 'Underlying Tokens': 'So11111111111111111111111111111111111111112', 'Image': None, 'App Link': None}
{'Project': 'jito-liquid-staking', 'Symbol': 'JITOSOL', 'APY': '5.17', 'APY Base': '5.17', 'APY Reward': '0', 'Reward Tokens': '', 'Pool ID': '0e7d0722-9054-4907-8593-567b353c0900', 'APY % 1D': '-0.01', 'APY % 7D': '-0.1', 'APY % 30D': '0.06', 'Stablecoin': 'False', 'APY Base 7D': '0', 'TVL USD': '744980856', 'Underlying Tokens': 'So11111111111111111111111111111111111111112', 'Image': None, 'App Link': None}
{'Project': 'blackrock-buidl', 'Symbol': 'BUIDL', 'APY': '3.5379', 'APY Base': '3.5379', 'APY Reward': '0', 'Reward Tokens': '', 'Pool ID': '590d770e-ed5d-4c8d-ad96-

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("solana_pools.db")

df = pd.read_sql_query("SELECT * FROM pools_enriched LIMIT 5", conn)
print(df.to_string(index=False))

conn.close()

            Project  Symbol     APY APY Base APY Reward                                Reward Tokens                              Pool ID APY % 1D APY % 7D APY % 30D Stablecoin APY Base 7D   TVL USD                            Underlying Tokens Pool Address Image App Link
 binance-staked-sol   BNSOL  4.7782   4.7782          0                                              9e709e57-84eb-496b-82ce-2e8f6a17db1b -0.06135  -0.1271  -0.31119      False           0 756970273  So11111111111111111111111111111111111111112         None  None     None
jito-liquid-staking JITOSOL    5.17     5.17          0                                              0e7d0722-9054-4907-8593-567b353c0900    -0.01     -0.1      0.06      False           0 744980856  So11111111111111111111111111111111111111112         None  None     None
    blackrock-buidl   BUIDL  3.5379   3.5379          0                                              590d770e-ed5d-4c8d-ad96-5178c2072295   -3e-05  0.05572   0.02415       True        

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("solana_pools.db")

df = pd.read_sql_query("SELECT COUNT(DISTINCT Project) FROM pools_enriched", conn)
print(df.to_string(index=False))

conn.close()

 COUNT(DISTINCT Project)
                      52


In [7]:
import sqlite3
conn = sqlite3.connect("solana_pools.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute('SELECT Project, Symbol, "Image", "App Link" FROM pools_enriched LIMIT 5')
for row in cur.fetchall():
    print(dict(row))

{'Project': 'binance-staked-sol', 'Symbol': 'BNSOL', 'Image': 'https://icons.llamao.fi/icons/protocols/binance-staked-sol', 'App Link': 'https://www.binance.com/en/solana-staking'}
{'Project': 'jito-liquid-staking', 'Symbol': 'JITOSOL', 'Image': 'https://icons.llamao.fi/icons/protocols/jito-liquid-staking', 'App Link': 'https://jito.network'}
{'Project': 'blackrock-buidl', 'Symbol': 'BUIDL', 'Image': 'https://icons.llamao.fi/icons/protocols/blackrock-buidl', 'App Link': 'https://securitize.io/'}
{'Project': 'jupiter-lend', 'Symbol': 'USDC', 'Image': 'https://icons.llamao.fi/icons/protocols/jupiter-lend', 'App Link': 'https://jup.ag/?ref=f6y1ryr2snn3'}
{'Project': 'jupiter-staked-sol', 'Symbol': 'JUPSOL', 'Image': 'https://icons.llamao.fi/icons/protocols/jupiter-staked-sol', 'App Link': 'https://jup.ag/?ref=f6y1ryr2snn3'}


In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("solana_pools.db")

query = """
SELECT
    Project AS Protocol,
    "Pool Address",
    Symbol AS Asset,
    APY,
    "APY Base" AS "Base APY",
    "APY Reward" AS "Reward APY",
    "TVL USD" AS "TVL ($)",
    "Reward Tokens",
    "APY % 1D" AS "APY (1D)",
    "APY % 7D" AS "APY (7D)",
    "APY % 30D" AS "APY (30D)",
    Image,
    "App Link"
FROM pools_enriched
LIMIT 5
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

conn.close()

           Protocol Pool Address   Asset     APY Base APY Reward APY   TVL ($)                                Reward Tokens APY (1D) APY (7D) APY (30D)                                                       Image                                  App Link
 binance-staked-sol         None   BNSOL 4.73901  4.73901          0 748945575                                                     0 -0.02491  -0.33015  https://icons.llamao.fi/icons/protocols/binance-staked-sol https://www.binance.com/en/solana-staking
jito-liquid-staking         None JITOSOL    5.17     5.17          0 730859412                                                     0        0     -0.29 https://icons.llamao.fi/icons/protocols/jito-liquid-staking                      https://jito.network
    blackrock-buidl         None   BUIDL 3.55577  3.55577          0 654448556                                               0.01886  0.05471   0.04734     https://icons.llamao.fi/icons/protocols/blackrock-buidl                    https:/